# Snowflake Table Functions — Complete Notes
*Co-authored with CoCo*

## 1. What is a Table Function?

A **Table Function** is a function that returns a **set of rows** (a tabular result) rather than a single scalar value. You use it in the `FROM` clause of a query, just like a table or view.

### Key Characteristics
- Returns **zero or more rows** with one or more columns
- Called using the `TABLE()` wrapper in the `FROM` clause
- Can accept scalar arguments (constants, columns, expressions)
- Can be **system-defined** (built-in) or **user-defined** (UDTF)

### Syntax
```sql
SELECT *
FROM TABLE( function_name( arguments ) );
```

### Table Function vs Scalar Function vs Aggregate Function

| Feature | Scalar Function | Aggregate Function | Table Function |
|---------|----------------|-------------------|----------------|
| Returns | Single value | Single value from group | Multiple rows |
| Used in | SELECT, WHERE, etc. | SELECT with GROUP BY | FROM clause |
| Example | `UPPER('hello')` | `SUM(amount)` | `FLATTEN(input)` |
| Output | One value per input row | One value per group | Many rows per call |

## 2. How Many Table Functions Exist in Snowflake?

There is **no fixed number** — Snowflake continuously adds new built-in functions with each release.

### Key Facts
- Snowflake provides **hundreds of built-in functions**, many of which are table functions
- Listed under the **System-Defined Table Functions** section of official documentation
- Users can also create **User-Defined Table Functions (UDTFs)**, so total count per account varies
- Snowflake does not publish a single static count

### How to Discover Available Table Functions

Refer :- https://docs.snowflake.com/en/sql-reference/functions-table

```sql
-- Show all functions (scalar)
SHOW FUNCTIONS;
```

## 3. Categories of Built-in Table Functions

### A. Data Transformation Functions
| Function | Purpose |
|----------|--------|
| `FLATTEN()` | Explodes semi-structured data (VARIANT, ARRAY, OBJECT) into rows |
| `SPLIT_TO_TABLE()` | Splits a string by delimiter into rows |
| `STRTOK_SPLIT_TO_TABLE()` | Tokenizes a string into rows |
| `LATERAL FLATTEN()` | Combines LATERAL join with FLATTEN for correlated expansion |

### B. Data Generation Functions
| Function | Purpose |
|----------|--------|
| `GENERATOR()` | Generates a specified number of rows (useful for sequences, test data) |

### C. Account Usage / History Functions
| Function | Purpose |
|----------|--------|
| `QUERY_HISTORY()` | Returns query history for the account |
| `QUERY_HISTORY_BY_SESSION()` | Query history filtered by session |
| `QUERY_HISTORY_BY_USER()` | Query history filtered by user |
| `QUERY_HISTORY_BY_WAREHOUSE()` | Query history filtered by warehouse |
| `COPY_HISTORY()` | Returns COPY INTO load history |
| `LOGIN_HISTORY()` | Returns login attempts |
| `LOGIN_HISTORY_BY_USER()` | Login history filtered by user |
| `TASK_HISTORY()` | Returns task execution history |
| `PIPE_USAGE_HISTORY()` | Returns Snowpipe usage |
| `WAREHOUSE_METERING_HISTORY()` | Returns warehouse credit usage |
| `DATABASE_STORAGE_USAGE_HISTORY()` | Database storage over time |
| `STAGE_STORAGE_USAGE_HISTORY()` | Stage storage over time |
| `REPLICATION_USAGE_HISTORY()` | Replication credit usage |
| `DATA_TRANSFER_HISTORY()` | Data transfer usage |

### D. Data Sharing & Listing Functions
| Function | Purpose |
|----------|--------|
| `VALIDATE()` | Validates files loaded via COPY INTO (returns errors) |

### E. Result Scanning
| Function | Purpose |
|----------|--------|
| `RESULT_SCAN()` | Converts the result of a previous query (by ID) into a table |

## 4. Deep Dive: Key Table Functions

---

### 4.1 FLATTEN()

The most commonly used table function. Explodes semi-structured data into relational rows.

**Syntax:**
```sql
SELECT *
FROM TABLE( FLATTEN( input => <expression>
                     [, path => '<path>']
                     [, outer => TRUE|FALSE]
                     [, recursive => TRUE|FALSE]
                     [, mode => 'OBJECT'|'ARRAY'|'BOTH'] ) );
```

**Parameters:**
| Parameter | Description | Default |
|-----------|------------|--------|
| `input` | VARIANT, OBJECT, or ARRAY to flatten | Required |
| `path` | Path to element within input to flatten | `''` (root) |
| `outer` | If TRUE, generates row even for zero-element input | `FALSE` |
| `recursive` | If TRUE, flattens nested structures recursively | `FALSE` |
| `mode` | What to flatten: OBJECT, ARRAY, or BOTH | `'BOTH'` |

**Output Columns:**
| Column | Description |
|--------|------------|
| `SEQ` | Sequence number (unique per input row) |
| `KEY` | Key for OBJECT elements, index for ARRAY |
| `PATH` | Path to this element |
| `INDEX` | Array index (NULL for objects) |
| `VALUE` | The value at this element |
| `THIS` | The element being flattened |

**Examples:**
```sql
-- Flatten an array column
SELECT t.id, f.value::STRING AS tag
FROM my_table t,
     TABLE(FLATTEN(input => t.tags)) f;

-- Flatten nested JSON with path
SELECT f.value:name::STRING AS item_name,
       f.value:price::NUMBER AS item_price
FROM orders o,
     TABLE(FLATTEN(input => o.order_data, path => 'items')) f;

-- OUTER FLATTEN (keep rows with empty/null arrays)
SELECT t.id, f.value::STRING AS tag
FROM my_table t,
     TABLE(FLATTEN(input => t.tags, outer => TRUE)) f;

-- Recursive flatten
SELECT *
FROM TABLE(FLATTEN(input => PARSE_JSON('{"a":{"b":[1,2,3]}}'),
                   recursive => TRUE));
```

### 4.2 GENERATOR()

Generates rows without requiring an underlying table. Useful for creating sequences, test data, or date ranges.

**Syntax:**
```sql
SELECT ...
FROM TABLE( GENERATOR( ROWCOUNT => <n> ) );

-- OR time-based generation
SELECT ...
FROM TABLE( GENERATOR( TIMELIMIT => <seconds> ) );
```

**Examples:**
```sql
-- Generate 100 rows
SELECT SEQ4() AS row_num
FROM TABLE(GENERATOR(ROWCOUNT => 100));

-- Generate a date sequence
SELECT DATEADD(DAY, SEQ4(), '2024-01-01'::DATE) AS date_value
FROM TABLE(GENERATOR(ROWCOUNT => 365));

-- Generate random test data
SELECT
    SEQ4() AS id,
    RANDSTR(10, RANDOM()) AS random_name,
    UNIFORM(1, 1000, RANDOM())::NUMBER(10,2) AS amount
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

-- Using with ROW_NUMBER
SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS seq_num
FROM TABLE(GENERATOR(ROWCOUNT => 50));
```

**Key Notes:**
- `SEQ1()/SEQ2()/SEQ4()/SEQ8()` generates sequence numbers of n bytes and sequence will repeats if all combination are exceeded.
- Use `ROWCOUNT` for exact number of rows
- Use `TIMELIMIT` to generate rows for a specific duration

### 4.3 SPLIT_TO_TABLE() and STRTOK_SPLIT_TO_TABLE()

Split strings into rows based on delimiters.

**SPLIT_TO_TABLE Syntax:**
```sql
SELECT *
FROM TABLE( SPLIT_TO_TABLE( <string>, <delimiter> ) );
```

**Output Columns:** `SEQ`, `INDEX`, `VALUE`

**STRTOK_SPLIT_TO_TABLE Syntax:**
```sql
SELECT *
FROM TABLE( STRTOK_SPLIT_TO_TABLE( <string>, <delimiters> ) );
```

**Key Difference:**
- `SPLIT_TO_TABLE`: Splits on a single delimiter string
- `STRTOK_SPLIT_TO_TABLE`: Each character in delimiter string is treated as a separate delimiter

**Examples:**
```sql
-- Split comma-separated values
SELECT t.id, s.VALUE AS email
FROM users t,
     TABLE(SPLIT_TO_TABLE(t.email_list, ',')) s;

-- Split on multiple delimiters (comma, semicolon, pipe)
SELECT VALUE
FROM TABLE(STRTOK_SPLIT_TO_TABLE('a,b;c|d', ',;|'));
-- Returns: a, b, c, d as separate rows

-- Using SPLIT_TO_TABLE with TRIM
SELECT TRIM(s.VALUE) AS clean_value
FROM TABLE(SPLIT_TO_TABLE('apple, banana, cherry', ',')) s;
```

### 4.4 RESULT_SCAN()

Converts the result of a previously executed query into a table that can be queried.

**Syntax:**
```sql
SELECT *
FROM TABLE( RESULT_SCAN( '<query_id>' ) );

-- Or use LAST_QUERY_ID()
SELECT *
FROM TABLE( RESULT_SCAN( LAST_QUERY_ID() ) );
```

**Examples:**
```sql
-- Run a SHOW command, then query its results
SHOW TABLES IN SCHEMA my_db.my_schema;

SELECT "name", "rows", "bytes"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "rows" > 1000000;

-- Query results of SHOW WAREHOUSES
SHOW WAREHOUSES;

SELECT "name", "size", "state"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "state" = 'STARTED';
```

**Key Notes:**
- Column names from SHOW commands are **lowercase and quoted**
- Results persist for 24 hours
- Works with any query, not just SHOW commands
- Useful for post-processing DDL/metadata commands

### 4.5 VALIDATE()

Returns errors from a previous COPY INTO operation. Essential for debugging data loading issues.

**Syntax:**
```sql
SELECT *
FROM TABLE( VALIDATE( <table_name>, JOB_ID => '<query_id>' ) );

-- Or use _last to reference last COPY
SELECT *
FROM TABLE( VALIDATE( my_table, JOB_ID => '_last' ) );
```

**Output Columns:** `ERROR`, `FILE`, `LINE`, `CHARACTER`, `BYTE_OFFSET`, `CATEGORY`, `CODE`, `SQL_STATE`, `COLUMN_NAME`, `ROW_NUMBER`, `ROW_START_LINE`, `REJECTED_RECORD`

**Example:**
```sql
-- Load data (may have errors with ON_ERROR = 'CONTINUE')
COPY INTO my_table
FROM @my_stage/data.csv
ON_ERROR = 'CONTINUE';

-- Check what errors occurred
SELECT *
FROM TABLE(VALIDATE(my_table, JOB_ID => '_last'));
```

### 4.6 Account Usage History Functions

These functions live in the `INFORMATION_SCHEMA` of each database and provide operational metadata.

```sql
-- Query history for last hour
SELECT *
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    DATE_RANGE_START => DATEADD(HOUR, -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 100
));

-- Query history by warehouse
SELECT *
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_WAREHOUSE(
    WAREHOUSE_NAME => 'COMPUTE_WH',
    DATE_RANGE_START => DATEADD(DAY, -1, CURRENT_TIMESTAMP())
));

-- Copy history for a table
SELECT *
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => 'MY_TABLE',
    START_TIME => DATEADD(DAY, -7, CURRENT_TIMESTAMP())
));

-- Login history
SELECT *
FROM TABLE(INFORMATION_SCHEMA.LOGIN_HISTORY(
    TIME_RANGE_START => DATEADD(DAY, -1, CURRENT_TIMESTAMP())
));

-- Warehouse metering (credits used)
SELECT *
FROM TABLE(INFORMATION_SCHEMA.WAREHOUSE_METERING_HISTORY(
    DATE_RANGE_START => DATEADD(DAY, -30, CURRENT_TIMESTAMP())
));
```

**Important Notes:**
- These functions are accessed via `INFORMATION_SCHEMA` of a database
- They have a **latency** (data may lag by a few minutes to hours)
- For longer history, use `SNOWFLAKE.ACCOUNT_USAGE` views instead

## 5. LATERAL Joins with Table Functions

The `LATERAL` keyword allows a table function to reference columns from preceding tables in the FROM clause (correlated subquery behavior).

**Syntax:**
```sql
SELECT t.*, f.*
FROM my_table t,
     LATERAL TABLE_FUNCTION(t.column) f;

-- Explicit LATERAL keyword (equivalent to comma syntax above)
SELECT t.*, f.*
FROM my_table t
LATERAL TABLE(FLATTEN(input => t.json_col)) f;
```

**Key Concept:** In Snowflake, the comma syntax before `TABLE(FLATTEN(...))` **implicitly** acts as a LATERAL join. Both forms below are equivalent:

```sql
-- Implicit LATERAL (comma syntax)
SELECT t.id, f.value
FROM my_table t,
     TABLE(FLATTEN(input => t.data)) f;

-- Explicit LATERAL
SELECT t.id, f.value
FROM my_table t
LATERAL TABLE(FLATTEN(input => t.data)) f;
```

**When to use OUTER => TRUE:**
```sql
-- Without OUTER: rows with NULL/empty arrays are dropped
-- With OUTER: rows with NULL/empty arrays are preserved (with NULLs)
SELECT t.id, f.value
FROM my_table t,
     TABLE(FLATTEN(input => t.tags, outer => TRUE)) f;
```

## 6. User-Defined Table Functions (UDTFs)

You can create your own table functions in Snowflake using SQL, JavaScript, Python, or Java.

---

### 6.1 SQL UDTF

```sql
CREATE OR REPLACE FUNCTION sales_by_region(region_name VARCHAR)
  RETURNS TABLE(product VARCHAR, total_sales NUMBER)
AS
$$
  SELECT product, SUM(amount)
  FROM sales
  WHERE region = region_name
  GROUP BY product
$$;

-- Usage
SELECT *
FROM TABLE(sales_by_region('WEST'));
```

---

### 6.2 JavaScript UDTF

```sql
CREATE OR REPLACE FUNCTION js_split(input_str VARCHAR, delimiter VARCHAR)
  RETURNS TABLE(part VARCHAR)
  LANGUAGE JAVASCRIPT
AS $$
{
  processRow: function(row, rowWriter, context) {
    var parts = row.INPUT_STR.split(row.DELIMITER);
    for (var i = 0; i < parts.length; i++) {
      rowWriter.writeRow({PART: parts[i]});
    }
  }
}
$$;

-- Usage
SELECT *
FROM TABLE(js_split('hello-world-foo', '-'));
```

---

### 6.3 Python UDTF

```sql
CREATE OR REPLACE FUNCTION py_range(start_val INT, end_val INT)
  RETURNS TABLE(num INT)
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.8'
  HANDLER = 'RangeGenerator'
AS $$
class RangeGenerator:
    def process(self, start_val, end_val):
        for i in range(start_val, end_val + 1):
            yield (i,)
$$;

-- Usage
SELECT *
FROM TABLE(py_range(1, 10));
```

**Python UDTF Structure:**
- Must define a **handler class** with a `process` method
- `process()` receives one row's arguments and **yields** tuples (one per output row)
- Optional `__init__` for setup and `end_partition` for final output

---

### 6.4 Java UDTF

```sql
CREATE OR REPLACE FUNCTION java_repeat(input VARCHAR, times INT)
  RETURNS TABLE(repeated VARCHAR)
  LANGUAGE JAVA
  HANDLER = 'RepeatHandler'
AS $$
import java.util.stream.Stream;

class RepeatHandler {
  public static Class getOutputClass() {
    return OutputRow.class;
  }
  
  public static class OutputRow {
    public String REPEATED;
    public OutputRow(String s) { this.REPEATED = s; }
  }
  
  public Stream<OutputRow> process(String input, int times) {
    return Stream.generate(() -> new OutputRow(input)).limit(times);
  }
}
$$;
```

## 7. UDTF with Partitions (Advanced)

UDTFs can process data in **partitions** using `PARTITION BY`, enabling group-level processing.

```sql
CREATE OR REPLACE FUNCTION running_total(amount NUMBER)
  RETURNS TABLE(cumulative NUMBER)
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.8'
  HANDLER = 'RunningTotal'
AS $$
class RunningTotal:
    def __init__(self):
        self.total = 0
    
    def process(self, amount):
        self.total += amount
        yield (self.total,)
    
    def end_partition(self):
        pass  # Optional: yield final summary rows
$$;

-- Usage with PARTITION BY
SELECT region, t.*
FROM sales s,
     TABLE(running_total(s.amount) OVER (PARTITION BY s.region ORDER BY s.sale_date)) t;
```

**Partition Processing Lifecycle:**
1. `__init__()` — Called once per partition (initialize state)
2. `process()` — Called once per row in the partition
3. `end_partition()` — Called after all rows processed (yield summary rows)

## 8. Performance Considerations

| Consideration | Best Practice |
|--------------|---------------|
| FLATTEN on large datasets | Filter data BEFORE flattening |
| GENERATOR with huge ROWCOUNT | Use reasonable limits; combine with LIMIT |
| RESULT_SCAN | Results expire after 24 hours |
| Recursive FLATTEN | Can explode row count; use specific PATHs when possible |
| UDTFs | Python/Java UDTFs have overhead vs SQL UDTFs |
| LATERAL joins | Ensure preceding filters reduce row count |

### Tips
```sql
-- BAD: Flatten everything then filter
SELECT f.value:name::STRING
FROM huge_table t,
     TABLE(FLATTEN(input => t.json_data, path => 'items')) f
WHERE t.status = 'ACTIVE';

-- BETTER: Filter first using a CTE
WITH active_records AS (
    SELECT json_data
    FROM huge_table
    WHERE status = 'ACTIVE'
)
SELECT f.value:name::STRING
FROM active_records t,
     TABLE(FLATTEN(input => t.json_data, path => 'items')) f;
```

## 9. Common Patterns & Real-World Use Cases

### Pattern 1: Explode JSON Arrays
```sql
-- Parse JSON array from a string column
SELECT
    o.order_id,
    f.value:product_id::INT AS product_id,
    f.value:quantity::INT AS quantity
FROM orders o,
     TABLE(FLATTEN(input => PARSE_JSON(o.items_json))) f;
```

### Pattern 2: Unpivot with FLATTEN
```sql
-- Convert key-value pairs from an OBJECT into rows
SELECT
    t.id,
    f.key AS attribute_name,
    f.value::STRING AS attribute_value
FROM my_table t,
     TABLE(FLATTEN(input => t.attributes)) f;
```

### Pattern 3: Generate Date Spine
```sql
-- Create a continuous date range
SELECT DATEADD(DAY, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, '2024-01-01'::DATE) AS calendar_date
FROM TABLE(GENERATOR(ROWCOUNT => 366));
```

### Pattern 4: Parse Delimited Tags
```sql
-- Normalize comma-separated tags into rows
SELECT
    p.post_id,
    TRIM(s.VALUE) AS tag
FROM posts p,
     TABLE(SPLIT_TO_TABLE(p.tags, ',')) s;
```

### Pattern 5: Audit Recently Failed Queries
```sql
SELECT query_id, error_message, query_text
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    DATE_RANGE_START => DATEADD(HOUR, -6, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 50
))
WHERE execution_status = 'FAIL';
```

## 10. Summary

| Topic | Key Takeaway |
|-------|-------------|
| Definition | Functions that return tabular results (rows + columns) |
| Invocation | Always via `TABLE(function_name(...))` in the FROM clause |
| Count | No fixed number; hundreds built-in + unlimited UDTFs |
| Most Used | `FLATTEN()`, `GENERATOR()`, `SPLIT_TO_TABLE()`, `RESULT_SCAN()` |
| LATERAL | Implicit with comma syntax; explicit with LATERAL keyword |
| UDTFs | Supported in SQL, JavaScript, Python, Java |
| Partitions | UDTFs can process partitioned data with `OVER (PARTITION BY ...)` |
| Performance | Filter early, avoid unnecessary recursive flattening |

---

**References:**
- [Snowflake Table Functions Documentation](https://docs.snowflake.com/en/sql-reference/functions-table)
- [FLATTEN Function](https://docs.snowflake.com/en/sql-reference/functions/flatten)
- [GENERATOR Function](https://docs.snowflake.com/en/sql-reference/functions/generator)
- [User-Defined Table Functions](https://docs.snowflake.com/en/developer-guide/udf/udf-overview)

## FLATTEN vs LATERAL FLATTEN — What's the Difference?

---

### Short Answer

> **There is NO functional difference** between `FLATTEN` and `LATERAL FLATTEN` in Snowflake when used with the comma-join syntax. The `LATERAL` keyword is **implicit** when you use `TABLE(FLATTEN(...))` with a comma join referencing a column from another table.

---

### The Concept of LATERAL

`LATERAL` is a SQL keyword that allows a **subquery or table function** in the FROM clause to reference columns from **preceding tables** in the same FROM clause. This is called a **correlated** reference.

Without LATERAL, each item in the FROM clause is independent and cannot "see" columns from other items.

---

### In Snowflake: LATERAL is Implicit for Table Functions

Snowflake **automatically applies LATERAL semantics** when you use `TABLE(FLATTEN(...))` with a comma join. All three forms below are **100% equivalent**:

```sql
-- Form 1: Comma syntax (LATERAL is implicit)
SELECT t.id, f.value
FROM my_table t,
     TABLE(FLATTEN(input => t.json_col)) f;

-- Form 2: Explicit LATERAL keyword
SELECT t.id, f.value
FROM my_table t,
     LATERAL TABLE(FLATTEN(input => t.json_col)) f;

-- Form 3: Explicit LATERAL with JOIN syntax
SELECT t.id, f.value
FROM my_table t
     JOIN LATERAL TABLE(FLATTEN(input => t.json_col)) f;
```

**All three produce the same execution plan and the same results.**

---

### Why Does LATERAL FLATTEN Exist Then?

| Reason | Explanation |
|--------|------------|
| **SQL Standard compliance** | LATERAL is part of the SQL standard (ISO SQL:2003+). Writing it explicitly is more portable and self-documenting. |
| **Readability** | Makes it clear that the table function depends on the preceding table's columns (correlated). |
| **Habit from other databases** | In PostgreSQL, Oracle, etc., LATERAL is required for correlated table expressions. Snowflake doesn't require it but accepts it. |
| **Non-table-function subqueries** | For LATERAL with regular subqueries (not table functions), you DO need the keyword explicitly. |

---

### When LATERAL Actually Matters (Non-Table-Function Case)

For **subqueries** (not table functions), `LATERAL` IS required if you want to reference outer columns:

```sql
-- This REQUIRES the LATERAL keyword (it's a subquery, not a table function)
SELECT t.id, sub.top_product
FROM customers t,
     LATERAL (
         SELECT product AS top_product
         FROM orders
         WHERE orders.customer_id = t.id  -- references t.id from outer table
         ORDER BY amount DESC
         LIMIT 1
     ) sub;

-- Without LATERAL, this would FAIL because the subquery can't see t.id
```

---

### Visual Comparison

```
┌─────────────────────────────────────────────────────────────────┐
│  TABLE(FLATTEN(t.col))    →  LATERAL is IMPLICIT (auto-applied) │
│  LATERAL FLATTEN(t.col)   →  LATERAL is EXPLICIT (same result)  │
│  LATERAL (SELECT ...)     →  LATERAL is REQUIRED (won't work    │
│                               without it for correlated refs)    │
└─────────────────────────────────────────────────────────────────┘
```

---

### Side-by-Side Example

Given this table:
```sql
CREATE TABLE employees (
    id INT,
    name VARCHAR,
    skills VARIANT  -- e.g., ["Python", "SQL", "Java"]
);
```

**Using FLATTEN (implicit LATERAL):**
```sql
SELECT e.name, f.value::STRING AS skill
FROM employees e,
     TABLE(FLATTEN(input => e.skills)) f;
```

**Using LATERAL FLATTEN (explicit LATERAL):**
```sql
SELECT e.name, f.value::STRING AS skill
FROM employees e,
     LATERAL TABLE(FLATTEN(input => e.skills)) f;
```

**Result (identical for both):**
```
| NAME  | SKILL  |
|-------|--------|
| Alice | Python |
| Alice | SQL    |
| Alice | Java   |
| Bob   | SQL    |
| Bob   | Go     |
```

---

### Summary Table

| Aspect | `TABLE(FLATTEN(...))` | `LATERAL TABLE(FLATTEN(...))` |
|--------|----------------------|-------------------------------|
| Behavior | Correlated (can reference outer cols) | Correlated (can reference outer cols) |
| Performance | Same | Same |
| Execution plan | Same | Same |
| LATERAL keyword | Implicit | Explicit |
| Readability | Shorter | More self-documenting |
| Portability | Snowflake-specific shorthand | Closer to SQL standard |
| Best practice | Fine for quick queries | Preferred for production/shared code |

---

### Key Takeaway

> In Snowflake, `FLATTEN` and `LATERAL FLATTEN` are **functionally identical**. The difference is purely syntactic. Use `LATERAL` explicitly when you want clarity in shared/production code, or when working with correlated subqueries (non-table-function cases where LATERAL is actually required).